<a href="https://colab.research.google.com/github/vinaykumar56/agentic-ai/blob/main/easylevel_problem1_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



```
#Install Packages
```



In [30]:
!pip install --upgrade langchain-openai
!pip install --upgrade langchain
!pip install --upgrade langchain-huggingface
!pip install --upgrade langchain-community
!pip install --upgrade langfuse
!pip install --upgrade fastmcp
!pip install --upgrade langchain-mcp-adapters

In [31]:
!pip install langgraph

Import packages

In [32]:
import os
import threading
import asyncio
from langchain_openai import OpenAI
from langchain_huggingface import HuggingFaceEndpoint
from langchain_core.tools import tool
from langchain_mcp_adapters.client import MultiServerMCPClient
from fastmcp import FastMCP
from langgraph.graph import StateGraph, START, END
from langchain.agents import create_agent

Configure Environment Variables

In [35]:
from getpass import getpass
HUGGINGFACEHUB_API_TOKEN = getpass('Enter OpenAI API Key:')
os.environ['HUGGINGFACEHUB_API_TOKEN'] = HUGGINGFACEHUB_API_TOKEN

Enter OpenAI API Key:··········


In [36]:
LANGFUSE_SECRET_KEY = getpass('Enter Langfuse API Key:')
LANGFUSE_PUBLIC_KEY = getpass('Enter Langfuse Public Key:')
os.environ['LANGFUSE_SECRET_KEY'] = LANGFUSE_SECRET_KEY
os.environ['LANGFUSE_PUBLIC_KEY'] = LANGFUSE_PUBLIC_KEY
os.environ['LANGFUSE_HOST']       = "https://cloud.langfuse.com"

WEATHER_API = getpass('Enter OpenWeatherMap API Key:')
os.environ['WEATHER_API'] = WEATHER_API

Enter Langfuse API Key:··········
Enter Langfuse Public Key:··········
Enter OpenWeatherMap API Key:··········


In [37]:
#Initialize the Gemma LLM from Hugging Face
llm = HuggingFaceEndpoint(
    repo_id="google/gemma-3-27b-it",
    task="text-generation",
    temperature=0.7,
    max_new_tokens=512,
    huggingfacehub_api_token=HUGGINGFACEHUB_API_TOKEN
)

Initialize Langfuse

In [38]:
from langfuse import Langfuse
langfuse = Langfuse()
from langfuse.langchain import CallbackHandler

langfuse = Langfuse(
    public_key=LANGFUSE_PUBLIC_KEY,
    secret_key=LANGFUSE_SECRET_KEY,
    host="https://cloud.langfuse.com"

)

auth_ok = langfuse.auth_check
if auth_ok:
    print("Authentication successful")
else:
    print("Authentication failed")

Authentication successful


In [39]:
import requests
from fastmcp import FastMCP # Added this import
mcp = FastMCP("DemoServer")
@mcp.tool
def get_weather(city):
  """This is a weather tool which takes the city name as input and return the weather details as output"""
  url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={os.environ.get("WEATHER_API")}&units=metric"

  response = requests.get(url)

  data = response.json()

  if response.status_code != 200:
        return {
            "success": False,
            "message": data.get("message", "Error fetching weather")
        }

  return {
        "success": True,
        "city": data["name"],
        "temperature": data["main"]["temp"],
        "weather": data["weather"][0]["description"],
        "humidity": data["main"]["humidity"],
        "wind_speed": data["wind"]["speed"]
    }


@mcp.tool()
def get_info(msg):
  """Fetch the information on the given topic"""
  information_prompt=f"Get the breif details in the few lines about the topic {msg}"
  response=llm.invoke(information_prompt)
  return response


@mcp.tool()
def backend_team(msg):
  """This tool will inform backend team to take an action based on the msg"""
  print("action to be taken by backend team : {msg}")
  return "backend team was informed about the action"


@mcp.tool()
def frontend_team(msg):
  """This tool will inform frontend team to take an action based on the msg"""
  print("action to be taken by frontend team : {msg}")
  return "backend team was informed about the action"

In [40]:
from langchain_core.messages import HumanMessage

@tool
def classifyer(text:str) -> str:
  """Generate classification of the user input using llm"""
  classify_prompt = f"Get the classification of the following text among options : info / action / summary - {text}"
  response = llm.invoke(HumanMessage(content = classify_prompt))
  return "user intent :" + response.content

Configure Environment Variables

In [41]:
def start_mcp():
  mcp.run(
      transport="streamable-http",
      host="127.0.0.10",
      port=8000,
      debug=True
  )
threading.Thread(target=start_mcp).start()

print("Local MCP server Running")

Local MCP server Running


Exception in thread Thread-6 (start_mcp):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_36811/957405230.py", line 2, in start_mcp
  File "/usr/local/lib/python3.12/dist-packages/fastmcp/server/mixins/transport.py", line 91, in run
    anyio.run(
  File "/usr/local/lib/python3.12/dist-packages/anyio/_core/_eventloop.py", line 77, in run
    return async_backend.run(func, args, {}, backend_options)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/anyio/_backends/_asyncio.py", line 2358, in run
    return runner.run(wrapper())
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/asyncio/runners.py", line 118, in run
    return self._loop.run_until_complete(task)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "

In [42]:
import asyncio
from langchain_mcp_adapters.client import MultiServerMCPClient
client = MultiServerMCPClient(
   { "connections": {
          "url":"http://127.0.0.10:8000",
          "transport":"streamable-http"
        }
   }
)

await asyncio.sleep(5) # Give the server some time to start
mcp_tools = await client.get_tools()

ExceptionGroup: unhandled errors in a TaskGroup (1 sub-exception)

In [43]:
tools=[classifyer]+mcp_tools

NameError: name 'mcp_tools' is not defined

In [44]:
from langchain.agents import create_agent

In [45]:
prompt = """ You are handling a customer service branch. You will receive cutomer feedbacks about the products.
From that feedback you need to identify the intent of the customer and based on that call the different tools.

for example :
  asking information about the product - intent is info
  complaining about product or need help for the product, means any kind of support required - intent is action
  asking summary of the weather - intent is summary

Answer:
<direct response>
"""

agent = create_agent(llm, tools, system_prompt=prompt)

NameError: name 'tools' is not defined

In [ ]:
query = input("Enter your query : ")
for event in agent.stream("input":query, stream_mode="values"):
  print(event)